## 1. OpenAI Embeddings

OpenAI Embeddings are a way to represent text as numerical vectors, capturing semantic meaning and relationships. They're super useful for tasks like search, clustering, recommendations, and more.

**How it works:**

* Text is converted into a vector (list of floating-point numbers)
* Similar texts have vectors that are close together in space
* Dissimilar texts have vectors that are far apart

**Use cases:**

- Search: Rank results by relevance to a query string
- Clustering: Group texts by similarity
- Recommendations: Suggest items with related text
- Anomaly detection: Identify outliers with little relatedness

**OpenAI's embedding models:**
- text-embedding-3-small: 1536 dimensions, ~62,500 pages per dollar
- text-embedding-3-large: 3072 dimensions, ~9,615 pages per dollar

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv() ## to load all the environment variables from .env file

True

In [3]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [6]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x799c2a488ad0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x799c2a489400>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [ ]:
text = "Building agents with LLM (large language model) as its core controller is a cool concept."

embed_vector = embeddings.embed_query(text) ## We can also pass a parameter named "dimension" with desired dimension value to get the embedding vector of that dimension. By default, it will return the embedding vector of dimension 1024.
embed_vector

[-0.0292136799544096,
 -0.0036870911717414856,
 0.01869615912437439,
 -0.003640536917373538,
 0.0006480342126451433,
 0.03176112473011017,
 0.006052043754607439,
 -0.0043314010836184025,
 -0.02980957366526127,
 0.03286352753639221,
 -0.0028751862701028585,
 -0.07043461501598358,
 0.0008244745549745858,
 -0.010011010803282261,
 -0.031075848266482353,
 -0.04165295884013176,
 0.002541858237236738,
 -0.032386813312768936,
 0.025623422116041183,
 0.007459842134267092,
 0.02742599882185459,
 -0.008007319644093513,
 0.019932638853788376,
 0.028662478551268578,
 -0.00284352945163846,
 -0.034353259950876236,
 -0.0017588170012459159,
 0.027887817472219467,
 0.0012234438909217715,
 0.02026038058102131,
 0.07532094419002533,
 -0.030807694420218468,
 -0.030599132180213928,
 0.01774272881448269,
 -0.026800312101840973,
 0.01924736052751541,
 0.01982835680246353,
 0.027992097660899162,
 0.007139549124985933,
 0.046807438135147095,
 -0.023210052400827408,
 -0.02198847196996212,
 -0.004297881852835417,

In [8]:
print(f"Length of the embedding vector: {len(embed_vector)}")

Length of the embedding vector: 1536


In [12]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("./speech.txt")
docs=loader.load()
docs

[Document(metadata={'source': './speech.txt'}, page_content='My dear countrymen and countrywomen,\nI want to tell you that our struggle is not just for independence, but for the freedom of humanity. We want to free ourselves from the shackles of oppression, exploitation, and injustice.\nLet us march forward with the principles of truth, non-violence, and love in our hearts. Let us work together to build a nation that is just, equitable, and prosperous for all.\nJai Hind!')]

In [14]:
## Spliting the text into smaller chunks using RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
text_chunks=text_splitter.split_documents(docs)
text_chunks

[Document(metadata={'source': './speech.txt'}, page_content='My dear countrymen and countrywomen,'),
 Document(metadata={'source': './speech.txt'}, page_content='I want to tell you that our struggle is not just for independence, but for the freedom of humanity.'),
 Document(metadata={'source': './speech.txt'}, page_content='of humanity. We want to free ourselves from the shackles of oppression, exploitation, and'),
 Document(metadata={'source': './speech.txt'}, page_content='exploitation, and injustice.'),
 Document(metadata={'source': './speech.txt'}, page_content='Let us march forward with the principles of truth, non-violence, and love in our hearts. Let us'),
 Document(metadata={'source': './speech.txt'}, page_content='our hearts. Let us work together to build a nation that is just, equitable, and prosperous for all.'),
 Document(metadata={'source': './speech.txt'}, page_content='Jai Hind!')]

In [16]:
## Calling Chroma vector database to create the vector store and add the text chunks to it.

from langchain_community.vectorstores import Chroma
## Already initialized the OpenAI embedding object in the variable named "embeddings" so we can directly pass it to the Chroma vector store.
db=Chroma.from_documents(text_chunks,embedding=embeddings) ## Two things will be passed to this method - 1. the chunks stored in text_chunks 2. the embedding object to get the embedding vector
db

In [17]:
query="Let us march forward with the principles of truth"

retrieved_res=db.similarity_search(query) ## Doing similarity search from DB using the query.

In [20]:
print(retrieved_res[0].page_content)

Let us march forward with the principles of truth, non-violence, and love in our hearts. Let us


## Steps Performed Till Now

1. We loaded the OpenAI API key from `.env` file
2. Initialized a Text loader to read the text from speech
3. Converted the loaded text to chunks to generate embedding vector
4. Initialized an object for generating embedding vector using OpenAIEmbeddings with `model="text-embedding-3-small"`
5. Post chunking, ChromaDB called for vector store with splitted chunks & embedding model 
6. Initialized a query for similarity_search from Chroma DB

## 2. Ollama Embeddings (Open Source)

Ollama embeddings let you generate vector representations from **open‑source** models running fully locally, ideal for semantic search and RAG without any external API. [docs.ollama](https://docs.ollama.com/capabilities/embeddings)

## What “Ollama Embeddings” are

- An embedding is a numeric vector that encodes the meaning of text so similar texts are close in vector space. [docs.spring](https://docs.spring.io/spring-ai/reference/api/embeddings/ollama-embeddings.html)
- Ollama exposes embedding models (e.g. `mxbai-embed-large`, `nomic-embed-text`, `embeddinggemma`) through local HTTP endpoints and CLI. [ollama](https://ollama.com/blog/embedding-models)
- Everything runs on your machine (macOS, Linux, Windows, Docker), so data never leaves your environment, which is useful for private RAG. [tigerdata](https://www.tigerdata.com/blog/finding-the-best-open-source-embedding-model-for-rag)

## Recommended open‑source embedding models in Ollama

Examples of popular open‑source embedding models you can `ollama pull` and run locally.

| Model name | Dim | Typical use case |
|---|---|---|
| `mxbai-embed-large` | 1024 | High‑quality semantic search and RAG over documents.  [ollama](https://ollama.com/blog/embedding-models) |
| `nomic-embed-text` | 768 | General‑purpose text embeddings, good tradeoff quality/speed.  [collabnix](https://collabnix.com/ollama-embedded-models-the-complete-technical-guide-to-local-ai-embeddings-in-2025/) |
| `snowflake-arctic-embed` | 1024 | Code and technical docs similarity/search.  [collabnix](https://collabnix.com/ollama-embedded-models-the-complete-technical-guide-to-local-ai-embeddings-in-2025/) |
| `all-minilm` (GGUF variants) | 384 | Lightweight, fast embeddings for large‑scale or low‑resource setups.  [docs.spring](https://docs.spring.io/spring-ai/reference/api/embeddings/ollama-embeddings.html) |
| `embeddinggemma` | 300M params | Small Google Gemma‑based embedding model, good for local/dev.  [docs.ollama](https://docs.ollama.com/capabilities/embeddings) |

In [38]:
from langchain_community.embeddings import OllamaEmbeddings

embedding_ollama = OllamaEmbeddings(
    model='embeddinggemma:latest'
    )

In [54]:
embedding_ollama_vector = embedding_ollama.embed_documents([
    "Ollama Embeddings is an open-source solution for generating text embeddings",
    "Which are numerical representations of text that capture semantic meaning",
])

In [56]:
embedding_ollama_vector[0]

[-0.10559430718421936,
 -0.004187866114079952,
 -0.009468679316341877,
 -0.031207147985696793,
 0.005752503871917725,
 0.03103562444448471,
 0.005721224006265402,
 0.06767930090427399,
 0.028771691024303436,
 -0.04863555356860161,
 -0.050027478486299515,
 -0.03618032857775688,
 -0.009285097010433674,
 -0.033964529633522034,
 0.010622776113450527,
 0.0007292387308552861,
 0.006699285004287958,
 0.0362185537815094,
 -0.021027378737926483,
 -0.018645280972123146,
 0.05491847172379494,
 -0.005130831152200699,
 -0.07386808097362518,
 -0.022348036989569664,
 -0.04192201793193817,
 0.00029406300745904446,
 0.01340577658265829,
 -0.060156483203172684,
 0.021251730620861053,
 -0.025027846917510033,
 0.06863335520029068,
 0.016595037654042244,
 0.033615048974752426,
 -0.003493573749437928,
 0.03055390901863575,
 0.04828498139977455,
 -0.052573155611753464,
 -0.0203714556992054,
 -0.0001373279665131122,
 0.019212661311030388,
 0.0438334085047245,
 0.09620296210050583,
 -0.018174367025494576,
 0.0

In [58]:
print(f"Length of the Ollamaembedding vector: {len(embedding_ollama_vector[0])}")

Length of the Ollamaembedding vector: 768


In [59]:
## Reruning the embedding query to check the consistency of the embedding vector generated by the Ollama embedding model.

vec = embedding_ollama.embed_query("What is the open-source solution for generating text embeddings?")
vec

[-0.0958847776055336,
 -0.031269658356904984,
 0.0006506502395495772,
 -0.01396695151925087,
 0.0321209691464901,
 0.0364735946059227,
 0.022183557972311974,
 0.03715873137116432,
 0.022572187706828117,
 -0.037152595818042755,
 -0.03866232931613922,
 -0.023066677153110504,
 0.017382914200425148,
 -0.014157951809465885,
 0.027869923040270805,
 -0.03621206805109978,
 0.0018311168532818556,
 -0.0032485127449035645,
 -0.012359941378235817,
 0.02372833527624607,
 0.046382565051317215,
 0.025264333933591843,
 -0.03904802352190018,
 -0.018952392041683197,
 -0.037606216967105865,
 0.0031500656623393297,
 0.015360643155872822,
 -0.08559536933898926,
 0.02757294289767742,
 0.018444834277033806,
 0.03315800428390503,
 0.0058341259136796,
 0.02041528932750225,
 -0.0011042210971936584,
 0.019600186496973038,
 0.04856215417385101,
 -0.028036056086421013,
 -0.035794179886579514,
 0.0006679898360744119,
 0.029984761029481888,
 0.029449790716171265,
 0.04978412762284279,
 -0.046476785093545914,
 0.0305

## 3. Embedding Technique Using HuggingFace

In [60]:
load_dotenv()

True

In [61]:
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

Sentence Transformers on Hugging Face is a library that makes it easy to work with transformer models for sentence embeddings 🤖. You can use it to generate embeddings for sentences, paragraphs, or documents, and apply them to tasks like semantic search, clustering, or classification.

**Key Features:**

`Pre-trained Models:` **Access a wide range of models like BERT, RoBERTa, and more

`Easy-to-use API:`Simple interface for generating embeddings and computing similarity

`Customization:` Fine-tune models on your own data for specific tasks

**Use Cases:**

`Semantic Search:` Find similar sentences or documents

`Clustering:` Group texts by semantic meaning

`Text Classification:` Classify texts into categories

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [2]:
text = "Ollama Embeddings is an open-source solution for generating text embeddings"

query_vector = embeddings.embed_query(text)
query_vector

[-0.03577866405248642,
 -0.03937027230858803,
 0.006699506193399429,
 0.023562265560030937,
 0.02328665181994438,
 0.02766815386712551,
 -0.06314568221569061,
 4.9329195462632924e-05,
 0.05626107379794121,
 -0.04322194308042526,
 0.041219357401132584,
 0.046724069863557816,
 0.01759938709437847,
 -0.002510887337848544,
 -0.011129067279398441,
 0.11177901923656464,
 0.05652060732245445,
 0.056658294051885605,
 -0.05743340775370598,
 -0.013657048344612122,
 0.015755964443087578,
 0.059925444424152374,
 0.13651327788829803,
 -0.04464580863714218,
 0.03749338164925575,
 0.009172501973807812,
 -0.038040656596422195,
 0.038076676428318024,
 0.07774242758750916,
 -0.022007934749126434,
 0.10386530309915543,
 -0.005613693036139011,
 0.00616831099614501,
 0.057045649737119675,
 0.01838807202875614,
 0.04569641873240471,
 -0.015354323200881481,
 0.028138095512986183,
 -0.0437813326716423,
 0.019523583352565765,
 0.05628196522593498,
 0.055841878056526184,
 -0.0010308257769793272,
 0.083005517721

In [4]:
print(f"Length of the HuggingFace embedding vector: {len(query_vector)}")

Length of the HuggingFace embedding vector: 384
